# Dynamic Fleet Dispatch & Vehicle Routing Analysis
### Reinforcement Learning, MCTS, and Operations Research Baselines

This notebook analyzes the dynamic fleet simulation results, decision latencies, SLA compliance, and dispatch heuristics under stochastic traffic conditions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print("Environment initialized.")

## 1. Load Simulation & Latency Datasets

In [ ]:
latency_df = pd.read_csv('../artifacts/metrics/latency_benchmark.csv')
steps_df = pd.read_csv('../artifacts/datasets/simulation_steps.csv')
requests_df = pd.read_csv('../artifacts/datasets/requests.csv')

display(latency_df)
print(f"Loaded {len(steps_df)} simulation steps and {len(requests_df)} requests.")

## 2. Decision Latency Analysis & SLA Budget Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(latency_df))
width = 0.2

ax.bar(x - 1.5*width, latency_df['p50'], width, label='Median (P50)', color='#2b5c8f')
ax.bar(x - 0.5*width, latency_df['mean'], width, label='Mean', color='#4682b4')
ax.bar(x + 0.5*width, latency_df['p95'], width, label='P95', color='#e67e22')
ax.bar(x + 1.5*width, latency_df['p99'], width, label='P99', color='#c0392b')

ax.axhline(45.0, color='#d35400', linestyle='--', linewidth=1.5, label='45ms SLA Budget')
ax.set_xticks(x)
ax.set_xticklabels(latency_df['name'], rotation=15, ha='right')
ax.set_yscale('log')
ax.set_ylabel('Latency (ms, log scale)')
ax.set_title('Inference Latency Across Dispatch Methods')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Fleet Utilization & Queue Dynamics Over Time

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(steps_df['time'], steps_df['pending_requests'], color='#8e44ad', linewidth=2)
ax1.set_ylabel('Pending Requests')
ax1.set_title('Simulation Dynamics: Pending Queue & Fleet Utilization')
ax1.grid(True, alpha=0.4)

ax2.plot(steps_df['time'], steps_df['fleet_utilization'] * 100, color='#27ae60', linewidth=2)
ax2.set_ylabel('Fleet Utilization (%)')
ax2.set_xlabel('Simulation Time (minutes)')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 4. Request Turnaround Time Distribution & SLA Performance

In [ ]:
delivered = requests_df[requests_df['status'] == 'DELIVERED']
on_time_ratio = (delivered['delivery_time'] <= delivered['deadline']).mean() * 100

plt.figure(figsize=(9, 5))
sns.histplot(delivered['turnaround_time'], bins=20, kde=True, color='#16a085')
plt.axvline(delivered['turnaround_time'].mean(), color='red', linestyle='--', label=f"Mean: {delivered['turnaround_time'].mean():.1f} min")
plt.title(f"Turnaround Time Distribution (SLA Compliance: {on_time_ratio:.1f}%)")
plt.xlabel('Turnaround Time (min)')
plt.ylabel('Delivered Requests')
plt.legend()
plt.tight_layout()
plt.show()